# 🤖 AI Engineering Fundamentals — Lezione 3
## Notebook Gruppo B

**ITS Novitas 4.0 | Martedì 26/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [3]:
GRUPPO = "B"
MEMBRI = ["Anna", "Lorenzo L.", "Stefano", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo B — Anna, Lorenzo L., Stefano


In [ ]:
#!pip install anthropic -q
#from google.colab import userdata
import anthropic, os
from dotenv import load_dotenv  #per usarlo in locale 

load_dotenv()                                          #per usarlo in locale 

#os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 17.7 MB/s eta 0:00:00
✅ Setup completato!


---
## 🎯 Tema del Gruppo B: Conversation History & Multi-turno

Esplorate come funziona la memory del chatbot attraverso la history,
e confrontate le tre strategie di gestione.

---
### Esercizio 1 — Con vs senza history *(guidato)*

La differenza più importante della lezione.
Stessa conversazione, due approcci: solo l'ultimo messaggio vs tutta la history.

In [5]:
# Esercizio 1 — il chatbot che dimentica vs quello che ricorda

domande = [
    "Mi chiamo Luca e studio AI Engineering a Sassari.",
    "Qual è la capitale della Sardegna?",
    "Come mi chiamo e cosa studio?",  # ← il test della memoria
]

# ── CHATBOT SENZA HISTORY ──────────────────────────────────────────
print("=" * 55)
print("CHATBOT SENZA HISTORY (manda solo l'ultimo messaggio)")
print("=" * 55)

for domanda in domande:
    # TODO: chiamate l'API con SOLO l'ultimo messaggio
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content": domanda}]  # solo l'ultimo
    )
    print(f"👤 {domanda}")
    print(f"🤖 {risposta.content[0].text}\n")

print()

# ── CHATBOT CON HISTORY ────────────────────────────────────────────
print("=" * 55)
print("CHATBOT CON HISTORY (manda tutta la lista)")
print("=" * 55)

history = []
for domanda in domande:
    history.append({"role": "user", "content": domanda})

    # TODO: chiamate l'API con TUTTA la history
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages= history  # tutta la lista
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    print(f"👤 {domanda}")
    print(f"🤖 {testo}\n")

''' Osservazione: cosa risponde alla terza domanda nei due casi?
Nel primo caso, senza history, risponde di non avere informazioni sulla conversazione precedente,
al contrario, quanto ha accesso a tutta la conversazione, risponde con le informazioni ottenute nelle prime domande. '''

CHATBOT SENZA HISTORY (manda solo l'ultimo messaggio)
👤 Mi chiamo Luca e studio AI Engineering a Sassari.
🤖 Ciao Luca! Piacere di conoscerti. 👋

Studi AI Engineering all'Università di Sassari – interessante! È un campo affascinante e in crescita. 

Che cosa ti attrae di più di questo percorso di studi? Stai già lavorando su progetti specifici o hai aree dell'AI che ti interessano particolarmente (machine learning, NLP, computer vision, ecc.)?

Se hai domande su AI, hai bisogno di aiuto con i tuoi studi o semplicemente vuoi parlare del tuo percorso, sono qui per aiutarti! 😊

👤 Qual è la capitale della Sardegna?
🤖 La capitale della Sardegna è **Cagliari**.

👤 Come mi chiamo e cosa studio?
🤖 Mi dispiace, ma non ho informazioni su chi sei tu. Non ho accesso a dati personali su di te né a conversazioni precedenti che potremmo aver avuto.

Se vuoi condividere il tuo nome e cosa studi, sono felice di saperlo! 😊


CHATBOT CON HISTORY (manda tutta la lista)
👤 Mi chiamo Luca e studio AI Engineer

' Osservazione: cosa risponde alla terza domanda nei due casi?\nNel primo caso, senza history, risponde di non avere informazioni sulla conversazione precedente,\nal contrario, quanto ha accesso a tutta la conversazione, risponde con le informazioni ottenute nelle prime domande. '

---
### Esercizio 2 — Truncation vs Sliding Window *(guidato)*

Fate una conversazione lunga (8 turni) con le due strategie.
Verificate cosa ricorda e cosa dimentica il chatbot in ogni caso.

In [6]:
# Esercizio 2 — truncation vs sliding window

MAX = 3  # manteniamo solo 3 turni per rendere visibile l'effetto

def chat_truncation(messaggio, history):
    history.append({"role": "user", "content": messaggio})

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    if len(history) > MAX * 2:
        history[:] = history[-MAX * 2:]
        print(f"  ✂️ History troncata a {MAX * 2} messaggi")

    return testo

def chat_sliding(messaggio, history):
    """Usa sliding window: la history completa rimane, manda solo una finestra."""
    history.append({"role": "user", "content": messaggio})

    # TODO: crea messaggi_da_inviare = history[-MAX*2:]
    messaggi_da_inviare = history[-MAX * 2:]

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=messaggi_da_inviare
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test con 6 turni
turni = [
    "Info 1: Mi chiamo Marco.",
    "Info 2: Lavoro a WiData.",
    "Info 3: Il mio sensore preferito è XS200.",
    "Domanda generica: cos'è il RAG?",
    "Altra domanda generica: cos'è lo streaming?",
    "Domanda generica: Cos'è l'IA?",
    "Spiegami cosa fanno i sensori ambientali",
    "Info 4: Abito a cagliari",
    "Info 5: Sono in ferie",

    "Come mi chiamo? Dove lavoro? Qual è il mio sensore preferito? Dove abito? Sono in ferie?",    #test memoria
]

hist_trunc = []
hist_slid  = []

print(f"{'Turno':<8} {'Truncation (history)':<25} {'Sliding (history)':<25}")
print("-" * 60)

for i, turno in enumerate(turni):
    r_t = chat_truncation(turno, hist_trunc)
    r_s = chat_sliding(turno, hist_slid)

    print(f"T{i+1:<7} len={len(hist_trunc):<20} len={len(hist_slid)}")

print("\n--- Risposte al test memoria ---")
print(f"Truncation: {r_t}")
print("################################################################")
print(f"Sliding:    {r_s}")


# Le due strategie danno risultati diversi? Quale ricorda di più?
# La truncation elimina i messaggi meno recenti, mentre la sliding non cancella la history completa ma manda solo gli ultimi messaggi al modello.
# Le 2 strategie danno circa gli stessi risultati se hanno lo stesso limite di quantità di messaggi


Turno    Truncation (history)      Sliding (history)        
------------------------------------------------------------
T1       len=2                    len=2
T2       len=4                    len=4
T3       len=6                    len=6
  ✂️ History troncata a 6 messaggi
T4       len=6                    len=8
  ✂️ History troncata a 6 messaggi
T5       len=6                    len=10
  ✂️ History troncata a 6 messaggi
T6       len=6                    len=12
  ✂️ History troncata a 6 messaggi
T7       len=6                    len=14
  ✂️ History troncata a 6 messaggi
T8       len=6                    len=16
  ✂️ History troncata a 6 messaggi
T9       len=6                    len=18
  ✂️ History troncata a 6 messaggi
T10      len=6                    len=20

--- Risposte al test memoria ---
Truncation: # Quello che So di Te 📋

Basandomi **solo su quello che mi hai detto in questa conversazione**:

| Domanda | Risposta | Fonte |
|---------|----------|-------|
| **Come ti chiami?** 

---
### Esercizio 3 — Summarization *(libero)*

Implementate la strategia più avanzata: quando la history
supera una soglia, chiedete a Claude di riassumerla
e usate il riassunto come contesto compresso.

In [7]:
# Esercizio 3 — summarization

SOGLIA_TURNI = 4  # quando superare questa soglia, riassumi

def riassumi_history(history):
    """Chiede a Claude di riassumere la conversazione in max 100 token."""
    testo_history = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history)
    # TODO: chiamate Claude per riassumere la conversazione
    # Il riassunto deve essere max 100 token e mantenere i punti essenziali
    riassunto = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[
            {"role": "user", "content": f"Riassumi questa conversazione:\n{testo_history}"}
        ]
    ).content[0].text
    return riassunto


def chat_con_summarization(messaggio, history):
    """Chatbot con summarization quando la history è troppo lunga."""
    history.append({"role": "user", "content": messaggio})

    # Costruisci il contesto: riassunto (se presente) + ultimi messaggi
    if len(history) > SOGLIA_TURNI * 2:
        # TODO: usa il riassunto come system prompt aggiuntivo
        riassunto = riassumi_history(history)
        system = f"Contesto della conversazione precedente: {riassunto}"
        messaggi_da_inviare = history[-4:]  # solo gli ultimi 2 turni
    else:
        system = ""
        messaggi_da_inviare = history

    # TODO: chiamate l'API con system e messaggi_da_inviare

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=messaggi_da_inviare,
        system=system
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test: create una conversazione lunga, riassumete, continuate

history = []

while True:
  domanda= input("Tu: ")
  if domanda == "esci":
    break
  risposta = chat_con_summarization(domanda, history)
  print(f"\nRisposta: {risposta}", "\n\n", "="*100)



# Verificate che dopo il riassunto il chatbot ricordi ancora le info iniziali
# ...

Tu: mi chiamo Stefano

Risposta: Piacere di conoscerti, Stefano! 👋

Come stai? C'è qualcosa in cui posso aiutarti? 

Tu: questa è la poesia che ho scritto: Tra i flussi digitali e il mare dei dati, dove i numeri scorrono non formattati, sorge WiData, con occhio sicuro, che traccia la rotta, che svela il futuro.  Non è solo codice, non è solo astratto, è dare alla logica la forza del fatto. Raccoglie i segnali, le tracce, i dettagli, riduce le ombre, cancella gli abbagli.  Come un faro moderno nel mare del cloud, lavora nel silenzio, lontano dal crowd. Connette i puntini, trasforma il sapere, in mappe precise, in chiare visioni da vedere.  Dall'algoritmo puro alla scelta aziendale, WiData modella il mondo virtuale: un ponte di bit, di idee e di ingegno, che lascia sul tempo un nitido segno.

Risposta: Stefano, è una bellissima poesia! 🌟

Mi piace molto come hai costruito l'inno a WiData. Ecco alcuni aspetti che funzionano davvero bene:

**Punti di forza:**
- **Metafore potenti**: il far

---
### Esercizio 4 — La strategia giusta per WiData *(libero)*

Un cliente di WiData fa mediamente 15 domande per sessione.
Ogni risposta è lunga circa 100 parole.

Quale strategia di gestione della history consigliate?
Implementatela e motivate la scelta con numeri concreti.

In [13]:
# Esercizio 4 — la strategia giusta per WiData


token_senza_strategia = 0
token_con_strategia = 0



# Simulate una sessione tipica WiData: 15 domande sui prodotti
domande_widata = [
    """Questa è una poesia su WiData:

      Tra i flussi digitali e il mare dei dati,
      dove i numeri scorrono non formattati,
      sorge WiData, con occhio sicuro,
      che traccia la rotta, che svela il futuro.

      Non è solo codice, non è solo astratto,
      è dare alla logica la forza del fatto.
      Raccoglie i segnali, le tracce, i dettagli,
      riduce le ombre, cancella gli abbagli.

      Come un faro moderno nel mare del cloud,
      lavora nel silenzio, lontano dal crowd.
      Connette i puntini, trasforma il sapere,
      in mappe precise, in chiare visioni da vedere.

      Dall'algoritmo puro alla scelta aziendale,
      WiData modella il mondo virtuale:
      un ponte di bit, di idee e di ingegno,
      che lascia sul tempo un nitido segno.

    """,

    "Mi chiamo Stefano",
    "Quali sensori offrite per ambienti esterni?",
    "Il sensore XS200 funziona con LoRaWAN?",
    "Qual è la durata della batteria?",
    "Come si installa il gateway GW500?",
    "La piattaforma Xplore ha le API REST?",
    "Cosa è il sistema Wodoszczelny RJ45?",
    "In che modo i vostri prodotti si occupano di non violare la privacy?",
    "Posso integrare Xplore con PowerBI?",
    "Quanto costa una licenza Xplore?",
    "I sensori funzionano offline?",
    "Qual è la copertura massima del gateway?",
    "I dati sono crittografati?",
    "Offrite assistenza tecnica in loco?",
    "Il sensore XS200 misura anche l'umidità?",
    "Come mi chiamo?",
    "Di cosa parlava la poesia su WiData?",
    "Ripetimi completamente la poesia su WiData"

]

# TODO: implementate la strategia che ritenete migliore
# Misurate: token totali, costo, qualità delle risposte

MAX = 10




def chat_senza_strategia(messaggio, history_test):

    global token_senza_strategia

    system = "Rispondi sempre in breve. "

    history_test.append({"role": "user", "content": messaggio})

    messaggi_da_inviare = history_test

    params = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 100,
    "messages": messaggi_da_inviare,
    "system": system
    }

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    count = client.messages.count_tokens(
      model="claude-haiku-4-5-20251001",
      messages= messaggi_da_inviare
    )
    token_senza_strategia += count.input_tokens

    history_test.append({"role": "assistant", "content": testo})

    return testo







def riassumi_history(history):

    global token_con_strategia

    da_riassumere = history[:-MAX]

    testo_history = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in da_riassumere
    )
    riassunto = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[
            {"role": "user", "content": f"Riassumi questa conversazione:\n{testo_history}"}
        ]
    ).content[0].text

    count = client.messages.count_tokens(
      model="claude-haiku-4-5-20251001",
      messages= [{"role": "user", "content": f"Riassumi questa conversazione:\n{testo_history}"}]
    )
    token_con_strategia += count.input_tokens
    return riassunto


def chat_WiData(messaggio, history):

    global token_con_strategia

    system = "Rispondi sempre in breve. "

    history.append({"role": "user", "content": messaggio})

    if len(history[:-1]) > MAX:
        riassunto = riassumi_history(history[:-1])
        system += f"Contesto della conversazione precedente: {riassunto}"
        messaggi_da_inviare = history[-(MAX+1):]
    else:
        messaggi_da_inviare = history

    params = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 100,
    "messages": messaggi_da_inviare,
    "system": system
    }



    risposta = client.messages.create(**params)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    count = client.messages.count_tokens(
      model="claude-haiku-4-5-20251001",
      messages= messaggi_da_inviare
    )

    token_con_strategia += count.input_tokens
    return testo



history = []
history_test = []

for domanda in domande_widata:
  print("Domanda: ", domanda)
  risposta_con_strategia = chat_WiData(domanda, history)
  risposta_senza_strategia = chat_senza_strategia(domanda, history_test)
  print("\nRisposta: ", risposta_con_strategia,"\n\n")
  print("="*100, "\n")



print("Token totali utilizzati in input con strategia: ", token_con_strategia)
print("Token totali utilizzati in input senza strategia: ", token_senza_strategia)

costo_per_sessione = token_con_strategia / 1_000_000 * 0.80
costo_1000_sessioni = costo_per_sessione * 1000
print(f"Costo per 1000 sessioni/mese usando la strategia: ${costo_1000_sessioni:.2f}")



# Confronto finale:
# Strategia scelta: sliding + summarization
# Motivazione: ricorda la conversazione della sessione di 15 domande dell'utente, e i messaggi precedenti vengono riassunti in modo da avere comunque un po' di contesto ma risparmiare token
# Token risparmiati vs nessuna gestione: Con il limite di messaggi "MAX = 10" con la strategia si risparmiano 172 token rispetto a nessuna strategia, ma al crescere della conversazione il risparmio di token della strategia diventa sempre più significativo
# Costo per 1000 sessioni/mese: $18.46

Domanda:  Questa è una poesia su WiData:

      Tra i flussi digitali e il mare dei dati,
      dove i numeri scorrono non formattati,
      sorge WiData, con occhio sicuro,
      che traccia la rotta, che svela il futuro.

      Non è solo codice, non è solo astratto,
      è dare alla logica la forza del fatto.
      Raccoglie i segnali, le tracce, i dettagli,
      riduce le ombre, cancella gli abbagli.

      Come un faro moderno nel mare del cloud,
      lavora nel silenzio, lontano dal crowd.
      Connette i puntini, trasforma il sapere,
      in mappe precise, in chiare visioni da vedere.

      Dall'algoritmo puro alla scelta aziendale,
      WiData modella il mondo virtuale:
      un ponte di bit, di idee e di ingegno,
      che lascia sul tempo un nitido segno.

    

Risposta:  Bella poesia! Cattura perfettamente l'essenza di WiData: trasformazione dei dati grezzi in insights chiari e actionable. Mi piacciono particolarmente:

- L'immagine del **faro nel cloud** (guida orie

---
## 📊 Preparate la presentazione (5 slide)

1. **Con vs senza history** — la differenza mostrata con i vostri risultati
2. **Il pattern corretto** — i 3 passi: aggiungi user, manda tutto, aggiungi assistant
3. **Truncation vs Sliding Window** — quando usa cosa, con i vostri dati
4. **Summarization** — come funziona e quando conviene
5. **La vostra raccomandazione per WiData** — con motivazione numerica

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*